# LCK 경기 결과 수집기 (Colab 실습용)

초보자를 위한 실습입니다. **Liquipedia 리그오브레전드 위키의 MediaWiki API**만 사용하고,
API 키 없이 접근 가능한 범위에서 아래 규칙을 지킵니다.

- 일반 요청 2초 간격, `action=parse` 요청 30초 간격
- 응답은 `lck_cache/`에 캐시해 재요청을 줄임
- 접근 거부(401/403/429), 인증 요구, 권한 부족이 나오면 **즉시 중단**
- 일시적 오류만 2회 재시도, 다른 수집처/가상 데이터 사용 금지

> **이용약관 주의**: Liquipedia는 MediaWiki API의 비편집 목적 사용을 공식 지원하지 않고,
> 데이터 목적은 승인된 LiquipediaDB API를 권장합니다. 이 노트북은 접근이 실제로 막히면
> 즉시 멈추도록 만들어진 **실습용** 코드입니다. 실제 서비스에서는 API 키를 신청하세요.

실행: 아래 셀을 위에서부터 순서대로 `Shift+Enter`로 실행하세요.
마지막에 `lck_match_results.json`이 현재 폴더(`/content`)에 저장되고 다운로드됩니다.

In [ ]:
!pip install -q requests beautifulsoup4
print('설치 완료')

## 1. 설정값

API 주소, 요청 간격, 캐시 위치, 수집 건수 등 자주 바꾸는 값입니다. `USER_AGENT`의 연락처는 다른 사람이 쓰면 본인 것으로 바꿔 주세요.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import quote, urljoin

import requests
from bs4 import BeautifulSoup

# ---- 자주 바꾸는 설정값 ----------------------------------------------------
API_URL = "https://liquipedia.net/leagueoflegends/api.php"
WIKI_BASE = "https://liquipedia.net"
KST = timezone(timedelta(hours=9))  # 한국 표준시(UTC+9)

# 다른 사람이 사용한다면 아래 연락처를 본인 것(예: GitHub 이슈 주소)으로 바꿔 주세요.
USER_AGENT = (
    "CC-LCK-Practice-Study/1.0 "
    "(https://github.com/kimhyungju-0472/CC-LCK-Practice; contact: GitHub Issues)"
)

MATCH_COUNT = 5                       # 수집할 경기 수
CACHE_DIR = Path("lck_cache")         # API 응답 캐시 폴더
OUTPUT_FILE = Path("lck_match_results.json")
CACHE_TTL_SECONDS = 24 * 60 * 60      # 캐시 유효 기간: 24시간
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 2                       # 일시적 오류 재시도 횟수
DATE_UNKNOWN = "날짜 확인 불가"

## 2. 오류 처리와 API 요청 관리

접근 제한은 즉시 멈추고, 네트워크 같은 일시적 오류만 2번 재시도합니다. 캐시와 요청 간격도 여기서 처리합니다.

In [ ]:
class CollectorError(Exception):
    """수집 중 사용자가 확인해야 하는 오류를 담는다."""

    def __init__(self, stage: str, message: str, cause: str, solution: str):
        super().__init__(message)
        self.stage = stage
        self.message = message
        self.cause = cause
        self.solution = solution


class AccessBlockedError(CollectorError):
    """인증/권한/요청 제한처럼 즉시 멈춰야 하는 오류."""

    def __init__(self, stage: str, message: str):
        super().__init__(
            stage=stage,
            message=message,
            cause="Liquipedia API가 접근을 거부했습니다. (인증 요구, 권한 부족, 요청 제한 등)",
            solution=(
                "요청 간격(일반 2초, parse 30초)을 지키고 캐시를 사용하세요. "
                "계속 거부되면 Liquipedia 관리자에게 문의하거나 승인된 LiquipediaDB API를 신청하세요. "
                "접근 제한을 우회하는 시도는 하지 마세요."
            ),
        )


def print_error_report(error: CollectorError) -> None:
    """오류 발생 단계/실제 메시지/원인/해결 방법을 한글로 출력한다."""
    print("\n" + "=" * 68)
    print("데이터 수집을 중단했습니다. (가짜 데이터는 만들지 않습니다)")
    print("=" * 68)
    print(f"1) 발생 단계     : {error.stage}")
    print(f"2) 실제 오류     : {error.message}")
    print(f"3) 원인          : {error.cause}")
    print(f"4) 해결 방법     : {error.solution}")
    print("=" * 68)


class RateLimiter:
    """API 종류별 최소 요청 간격을 지킨다."""

    # 이용약관: 일반 1 req/2s, action=parse 1 req/30s (여유초를 더함)
    MIN_INTERVAL = {"query": 2.2, "parse": 30.5}
    DENIED_MARKERS = ("auth", "permission", "forbidden", "blocked", "denied", "rate limit")

    def __init__(self):
        self.last_request = {"query": 0.0, "parse": 0.0}

    def wait(self, kind: str) -> None:
        if kind not in self.MIN_INTERVAL:
            kind = "query"
        gap = self.MIN_INTERVAL[kind] - (time.monotonic() - self.last_request[kind])
        if gap > 0:
            time.sleep(gap)
        self.last_request[kind] = time.monotonic()


class LiquipediaClient:
    """캐시와 재시도를 곁들인 아주 작은 API 클라이언트."""

    def __init__(self):
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        self.limiter = RateLimiter()
        self.session = requests.Session()  # 연결을 재사용한다(이용약관 권장)

    def _cache_path(self, params: dict) -> Path:
        key = json.dumps(params, sort_keys=True, ensure_ascii=False).encode("utf-8")
        return CACHE_DIR / (hashlib.sha256(key).hexdigest()[:20] + ".json")

    def _read_cache(self, path: Path):
        if not path.exists():
            return None
        try:
            saved = json.loads(path.read_text(encoding="utf-8"))
            if time.time() - saved.get("saved_at", 0) < CACHE_TTL_SECONDS:
                return saved["payload"]
        except (ValueError, KeyError, OSError):
            pass  # 깨진 캐시는 무시하고 새로 요청한다
        return None

    def _write_cache(self, path: Path, payload: dict) -> None:
        try:
            path.write_text(
                json.dumps({"saved_at": time.time(), "payload": payload}, ensure_ascii=False),
                encoding="utf-8",
            )
        except OSError:
            pass  # 캐시 저장 실패는 수집 자체를 막지 않는다

    def get(self, params: dict, stage: str, kind: str = "query") -> dict:
        """GET 요청을 보내고 JSON을 돌려준다. 접근 거부는 즉시 예외를 던진다."""
        request_params = dict(params, format="json", formatversion=2)
        cache_path = self._cache_path(request_params)
        cached = self._read_cache(cache_path)
        if cached is not None:
            print(f"  [캐시 사용] {stage}")
            return cached

        last_error = None
        for attempt in range(MAX_RETRIES + 1):
            self.limiter.wait(kind)
            try:
                response = self.session.get(
                    API_URL,
                    params=request_params,
                    headers={
                        "User-Agent": USER_AGENT,
                        "Accept-Encoding": "gzip, deflate",  # 응답 크기 줄이기
                        "Accept": "application/json",
                    },
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )
                response.raise_for_status()
                payload = response.json()
            except requests.HTTPError as exc:
                status = exc.response.status_code if exc.response is not None else "?"
                body = exc.response.text[:400] if exc.response is not None else ""
                detail = f"HTTP {status}: {exc} / 응답 앞부분: {body}"
                if status in (401, 403, 429):
                    raise AccessBlockedError(stage, detail) from exc
                if isinstance(status, int) and 500 <= status <= 599 and attempt < MAX_RETRIES:
                    last_error = detail
                    print(f"  [재시도 {attempt + 1}/{MAX_RETRIES}] 서버 오류: {detail[:120]}")
                    continue
                raise CollectorError(
                    stage,
                    detail,
                    "서버가 요청을 처리하지 못했습니다.",
                    "잠시 후 다시 실행해 보세요. 요청 간격은 자동으로 지켜집니다.",
                ) from exc
            except (requests.Timeout, requests.ConnectionError) as exc:
                last_error = str(exc)
                if attempt < MAX_RETRIES:
                    print(f"  [재시도 {attempt + 1}/{MAX_RETRIES}] 네트워크 오류: {exc}")
                    continue
                raise CollectorError(
                    stage,
                    str(exc),
                    "일시적인 네트워크 오류가 2회 재시도 후에도 해결되지 않았습니다.",
                    "인터넷 연결을 확인하고 잠시 후 다시 실행하세요.",
                ) from exc
            except ValueError as exc:
                raise CollectorError(
                    stage,
                    str(exc),
                    "API가 JSON 대신 다른 형식을 반환했습니다.",
                    "Liquipedia API 상태를 확인하고 잠시 후 다시 실행하세요.",
                ) from exc

            if isinstance(payload, dict) and "error" in payload:
                api_error = payload["error"]
                message = api_error.get("message", str(api_error)) if isinstance(api_error, dict) else str(api_error)
                if any(word in message.lower() for word in self.limiter.DENIED_MARKERS):
                    raise AccessBlockedError(stage, f"API 오류: {message}")
                raise CollectorError(
                    stage,
                    f"API 오류: {message}",
                    "API가 요청을 거부했습니다.",
                    "요청 파라미터와 이용약관을 확인하세요.",
                )

            self._write_cache(cache_path, payload)
            return payload

        raise CollectorError(
            stage,
            str(last_error),
            "재시도 후에도 응답을 받지 못했습니다.",
            "네트워크 상태를 확인하고 잠시 후 다시 실행하세요.",
        )

## 3. 가장 최근에 끝난 LCK 대회 찾기

`LCK/연도`와 `LCK/연도/Cup` 문서의 종료일(`edate`)을 확인해, 오늘 기준으로 이미 끝난 가장 최근 대회를 고릅니다.

In [ ]:
def get_wikitext(client: LiquipediaClient, title: str, stage: str) -> str:
    """문서의 위키 텍스트(원본 문법)를 가져온다. 없는 문서는 빈 문자열."""
    payload = client.get(
        {
            "action": "query",
            "titles": title,
            "prop": "revisions",
            "rvprop": "content",
            "rvslots": "main",
        },
        stage=stage,
    )
    pages = payload.get("query", {}).get("pages", [])
    if not pages:
        return ""
    content = pages[0].get("revisions", [{}])[0].get("slots", {}).get("main", {}).get("content", "")
    return content or ""


def find_latest_completed_tournament(client: LiquipediaClient):
    """오늘(KST) 기준으로 이미 끝난 가장 최근 LCK 대회를 찾는다."""
    today = datetime.now(KST).date()

    # LCK/2026, LCK/2026/Cup 처럼 '대회 대표 문서'만 후보로 본다.
    # 아카데미/통계/선수 문서는 제외된다.
    for year in range(today.year, today.year - 6, -1):
        candidates = []
        for suffix in ("", "/Cup"):
            title = f"LCK/{year}{suffix}"
            wikitext = get_wikitext(client, title, f"대회 확인 ({title})")
            if not wikitext:
                continue

            end_match = re.search(r"\|\s*edate\s*=\s*(\d{4}-\d{2}-\d{2})", wikitext)
            if not end_match:
                continue
            end_date = datetime.strptime(end_match.group(1), "%Y-%m-%d").date()
            if end_date > today:  # 아직 끝나지 않은 대회는 제외
                continue

            name_match = re.search(r"\|\s*name\s*=\s*([^\n|}]+)", wikitext)
            name = name_match.group(1).strip() if name_match else title.replace("/", " ")

            candidates.append(
                {
                    "name": name,
                    "page": title,
                    "end_date": end_date.isoformat(),
                    "url": f"{WIKI_BASE}/leagueoflegends/{quote(title.replace(' ', '_'))}",
                    "wikitext": wikitext,
                }
            )

        if candidates:
            latest = max(candidates, key=lambda item: item["end_date"])
            print(f"대회 선택: {latest['name']} (종료일 {latest['end_date']}, {latest['page']})")
            return latest

    raise CollectorError(
        "대회 선택",
        "최근 6년 안에서 종료일이 확인되는 LCK 대회를 찾지 못했습니다.",
        "위키 문서에 edate(종료일)가 없거나 아직 진행 중인 대회뿐입니다.",
        "위키 문서 구조를 확인하거나 날짜가 지난 뒤 다시 실행하세요.",
    )


def extract_stage_titles(tournament: dict) -> list[str]:
    """대회 문서의 탭 링크에서 경기 단계(하위 문서) 제목을 뽑는다."""
    links = re.findall(r"\|\s*link\d+\s*=\s*/([^\n|}]+)", tournament["wikitext"])
    stages = []
    for link in links:
        stage = link.strip().replace("_", " ")
        if not stage or stage.startswith("{"):
            continue
        title = f"{tournament['page']}/{stage}"
        if title not in stages:
            stages.append(title)
    return stages

## 4. 경기 결과 읽기

위키 문서를 사람이 보는 HTML로 렌더링받아, 날짜(KST), 팀, 점수, 승리팀, 경기 링크를 읽습니다. 날짜가 없으면 `날짜 확인 불가`로 표시합니다.

In [ ]:
def kst_date_from_timestamp(timestamp: str | None, timer_text: str) -> str:
    """경기 시각을 KST 날짜(YYYY-MM-DD)로 바꾼다. 확인 불가면 정해진 문구."""
    if timestamp:
        try:
            dt = datetime.fromtimestamp(int(timestamp), timezone.utc).astimezone(KST)
            return dt.strftime("%Y-%m-%d")
        except (TypeError, ValueError, OSError, OverflowError):
            pass

    # 데이터 속성이 없을 때: 화면에 보이는 'September 13, 2026 ... KST'를 사용
    match = re.search(r"([A-Z][a-z]+\s+\d{1,2},\s+\d{4})", timer_text)
    if match and "KST" in timer_text:
        try:
            return datetime.strptime(match.group(1), "%B %d, %Y").strftime("%Y-%m-%d")
        except ValueError:
            pass
    return DATE_UNKNOWN


def parse_matches(html: str, page_title: str) -> list[dict]:
    """action=parse가 돌려준 HTML에서 경기 결과를 읽는다."""
    soup = BeautifulSoup(html, "html.parser")
    results = []

    for block in soup.select("div.brkts-match"):
        # 한 경기는 팀 2개가 들어 있는 블록이다.
        entries = block.find_all("div", class_="brkts-opponent-entry", recursive=False)
        if len(entries) != 2:
            continue

        names, scores = [], []
        for entry in entries:
            name_tag = (
                entry.select_one(".team-name")
                or entry.select_one(".team-bracketname")
                or entry.select_one(".team-shortname")
            )
            score_tag = entry.select_one(".brkts-opponent-score-inner")
            names.append(name_tag.get_text(" ", strip=True) if name_tag else "")
            scores.append(score_tag.get_text(strip=True) if score_tag else "")

        # 이름/점수가 빠진 경기는 추측하지 않고 건너뛴다.
        if not all(names) or not all(scores):
            continue

        winner_index = None
        for index, entry in enumerate(entries):
            if entry.select_one(".brkts-opponent-win"):
                winner_index = index
                break
        if winner_index is None and scores[0] != scores[1]:
            # 승리 표시가 없으면 점수로 판단한다(추측이 아니라 명시된 점수 사용).
            winner_index = 0 if int(scores[0]) > int(scores[1]) else 1

        winner = names[winner_index] if winner_index is not None else "승리팀 확인 불가"

        timer = block.select_one(".timer-object")
        timer_text = timer.get_text(" ", strip=True) if timer else ""
        date_kst = kst_date_from_timestamp(timer.get("data-timestamp") if timer else None, timer_text)
        try:
            timestamp = int(timer["data-timestamp"]) if timer and timer.get("data-timestamp") else None
        except (TypeError, ValueError):
            timestamp = None

        match_link = block.select_one('a[href*="Match:"]')
        source_url = (
            urljoin(WIKI_BASE, match_link["href"])
            if match_link and match_link.get("href")
            else f"{WIKI_BASE}/leagueoflegends/{quote(page_title.replace(' ', '_'))}"
        )

        results.append(
            {
                "날짜_KST": date_kst,
                "팀": names,
                "최종점수": f"{scores[0]}:{scores[1]}",
                "승리팀": winner,
                "출처링크": source_url,
                "_timestamp": timestamp,
            }
        )
    return results


def collect_matches(client: LiquipediaClient, tournament: dict, stages: list[str]) -> list[dict]:
    """가장 최근 단계부터 경기를 모아 최신 5건을 고른다."""
    stage_titles = stages or [tournament["page"]]  # 하위 문서가 없으면 대회 문서 자체
    collected = []
    seen_links = set()

    # 탭 순서가 오래된 단계 -> 최신 단계이므로 뒤에서부터 본다.
    for stage in reversed(stage_titles):
        if len(collected) >= MATCH_COUNT:
            break
        payload = client.get(
            {"action": "parse", "page": stage, "prop": "text"},
            stage=f"경기 결과 확인 ({stage})",
            kind="parse",
        )
        html = payload.get("parse", {}).get("text", "")
        if not html:
            raise CollectorError(
                f"경기 결과 확인 ({stage})",
                f"페이지 HTML을 받지 못했습니다: {str(payload)[:300]}",
                "action=parse 응답에 text 항목이 없습니다.",
                "위키 문서가 삭제되었는지 확인하고 잠시 후 다시 실행하세요.",
            )

        matches = parse_matches(html, stage)
        print(f"  {stage}: 경기 {len(matches)}건 확인")
        for match in matches:
            if match["출처링크"] in seen_links:
                continue
            seen_links.add(match["출처링크"])
            collected.append(match)

    collected.sort(
        key=lambda item: (item["_timestamp"] is not None, item["_timestamp"] or 0),
        reverse=True,
    )
    return [{k: v for k, v in item.items() if not k.startswith("_")} for item in collected[:MATCH_COUNT]]

## 5. 수집 실행과 JSON 저장

결과를 화면에 출력하고 `lck_match_results.json`으로 저장합니다.

In [ ]:
def print_summary(result: dict) -> None:
    """결과를 사람이 보기 좋게 출력한다."""
    tournament = result["대회"]
    print("\n" + "=" * 68)
    print(f"대회: {tournament['이름']} (종료일 {tournament['종료일']})")
    print(f"수집: {result['수집건수']}건 / 요청 {result['요청건수']}건")
    print("=" * 68)
    for index, match in enumerate(result["경기결과"], 1):
        team1, team2 = match["팀"]
        print(
            f"{index}. {match['날짜_KST']} | {team1} {match['최종점수']} {team2} "
            f"| 승리: {match['승리팀']}"
        )
        print(f"   출처: {match['출처링크']}")
    print(f"\nJSON 저장: {OUTPUT_FILE.resolve()}")
    print("출처 표기: " + result["출처_표기"])


def main():
    """전체 수집 과정을 실행하고 결과 JSON을 저장한다."""
    try:
        client = LiquipediaClient()
        tournament = find_latest_completed_tournament(client)
        stages = extract_stage_titles(tournament)
        print(f"확인할 단계: {', '.join(stages) if stages else '(대회 문서 자체)'}")

        matches = collect_matches(client, tournament, stages)
        if len(matches) < MATCH_COUNT:
            print(f"[알림] 확인된 경기가 {len(matches)}건뿐입니다. 가짜 데이터는 채우지 않습니다.")

        result = {
            "수집시각_KST": datetime.now(KST).isoformat(timespec="seconds"),
            "요청건수": MATCH_COUNT,
            "수집건수": len(matches),
            "대회": {
                "이름": tournament["name"],
                "종료일": tournament["end_date"],
                "위키페이지": tournament["page"],
                "링크": tournament["url"],
            },
            "출처_표기": "자료 출처: Liquipedia League of Legends Wiki (CC BY-SA 3.0)",
            "이용약관_메모": (
                "MediaWiki API는 비편집 목적 사용이 공식 지원되지 않습니다. "
                "접근 제한 시 중단하며, 실제 서비스는 승인된 LiquipediaDB API를 사용하세요."
            ),
            "경기결과": matches,
        }
        OUTPUT_FILE.write_text(
            json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        )
        print_summary(result)
        return result
    except CollectorError as error:
        print_error_report(error)
        return None

## 6. Colab에서 실행하기

In [ ]:
def notebook_main():
    """Colab 노트북에서 호출하는 진입점."""
    return main()

In [ ]:
notebook_result = notebook_main()

# Colab에서는 JSON 파일을 내 컴퓨터로 다운로드합니다.
try:
    from google.colab import files
    files.download(str(OUTPUT_FILE))
except ImportError:
    print('Colab 환경이 아니면 현재 폴더에 JSON 파일이 저장됩니다.')

## 7. Discord 알림 (Webhook)

`lck_match_results.json`에 저장된 **실제 데이터만** 읽어 Discord로 보냅니다.
봇 계정이나 봇 토큰은 필요 없습니다.

**먼저 비밀 값 등록**
- 로컬: `DISCORD_WEBHOOK_URL` 환경 변수에 Discord 채널 웹후크 URL 설정
- Colab: 왼쪽 🔑(Secrets)에 `DISCORD_WEBHOOK_URL` 추가 후 **노트북 액세스 허용**

URL은 코드에 적지 않고, 출력/오류 메시지에도 노출되지 않습니다.
URL이 없으면 전송하지 않고 설정 방법만 안내합니다. 아래 셀을 순서대로 실행하세요.

### 7-1. Discord 설정값

웹훅 URL은 여기 쓰지 않습니다. Discord 제한값(본문 2000자, Embed 6000자)만 정의합니다.

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

import requests

DATA_FILE = Path("lck_match_results.json")
WEBHOOK_ENV_NAME = "DISCORD_WEBHOOK_URL"  # 값은 코드에 쓰지 않고 환경/Secrets에서 읽는다.

# Discord 제한값 (공식 문서 기준)
MAX_CONTENT_LENGTH = 2000   # 메시지 본문
MAX_EMBEDS = 10             # 메시지 1개당 Embed 개수
MAX_EMBED_TOTAL = 6000      # Embed 전체 글자 수 합계
MAX_TITLE_LENGTH = 256
MAX_DESCRIPTION_LENGTH = 4096
MAX_FIELD_NAME_LENGTH = 256
MAX_FIELD_VALUE_LENGTH = 1024

DISCORD_MAX_RETRIES = 2     # 일시적 네트워크/5xx 오류만 재시도
RETRY_DELAY_SECONDS = 2
DISCORD_TIMEOUT_SECONDS = 15
DISCORD_USER_AGENT = "CC-LCK-Practice-Discord/1.0 (https://github.com/kimhyungju-0472/CC-LCK-Practice)"
REDACTED = "[웹훅 URL 숨김]"

### 7-2. 웹훅 URL 읽기와 JSON 읽기

로컬은 환경 변수, Colab은 Secrets에서 `DISCORD_WEBHOOK_URL`을 읽습니다. URL이나 토큰은 출력하지 않습니다.

In [ ]:
class DiscordError(Exception):
    """사용자에게 보고해야 하는 Discord 알림 오류."""

    def __init__(self, stage: str, message: str, cause: str, solution: str):
        super().__init__(message)
        self.stage = stage
        self.message = message
        self.cause = cause
        self.solution = solution


def redact(text, webhook_url: str = "") -> str:
    """웹훅 URL/토큰을 어떤 출력에도 남기지 않는다."""
    safe = str(text)
    if webhook_url:
        safe = safe.replace(webhook_url, REDACTED)
    # requests 오류 메시지에 URL 경로가 들어가는 경우도 있다.
    safe = re.sub(r"https?://(?:canary\.)?discord(?:app)?\.com/api/webhooks/\S+", REDACTED, safe)
    return safe


def get_webhook_url() -> str:
    """로컬 환경 변수 -> Colab Secrets 순서로 웹훅 URL을 읽는다."""
    url = os.environ.get(WEBHOOK_ENV_NAME, "").strip()
    if url:
        return url

    # Colab에서는 좌측 🔑(Secrets)에 같은 이름으로 저장해 둔다.
    try:
        from google.colab import userdata
        url = (userdata.get(WEBHOOK_ENV_NAME) or "").strip()
        return url
    except Exception:
        # Colab이 아니거나 비밀이 아직 없으면 빈 문자열로 처리한다.
        return ""


def print_setup_guide() -> None:
    """웹훅 URL이 없을 때 설정 방법만 안내한다(가짜 URL은 만들지 않는다)."""
    print("전송 결과: 건너뜀 (실제 전송하지 않았습니다)")
    print(f"설정 방법 ({WEBHOOK_ENV_NAME}):")
    print("  - 로컬 PowerShell: $env:DISCORD_WEBHOOK_URL='여기에_웹훅_URL_입력'")
    print("  - 로컬 macOS/Linux: export DISCORD_WEBHOOK_URL='여기에_웹훅_URL_입력'")
    print("  - Colab: 왼쪽 🔑 Secrets에 DISCORD_WEBHOOK_URL 추가 후 '노트북 액세스' 허용")
    print("  - Discord: 채널 설정 → 연동 → 웹후크 → 새 웹후크 → URL 복사")


def discord_print_error_report(error: DiscordError, webhook_url: str = "") -> None:
    """오류 단계/실제 내용/원인/해결 방법을 한글로 보고한다."""
    print("\n" + "=" * 68)
    print("Discord 알림을 중단했습니다. (가짜 데이터/우회 시도 없음)")
    print("=" * 68)
    print(f"1) 발생 단계 : {redact(error.stage, webhook_url)}")
    print(f"2) 실제 내용 : {redact(error.message, webhook_url)}")
    print(f"3) 원인      : {redact(error.cause, webhook_url)}")
    print(f"4) 해결 방법 : {redact(error.solution, webhook_url)}")
    print("전송 결과: 실패")
    print("=" * 68)


def validate_webhook_url(url: str) -> None:
    """Discord 웹훅 형식인지 확인한다. URL 값은 절대 출력하지 않는다."""
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    allowed_hosts = {"discord.com", "discordapp.com", "canary.discord.com"}
    if (
        parsed.scheme != "https"
        or host not in allowed_hosts
        or not parsed.path.startswith("/api/webhooks/")
    ):
        raise DiscordError(
            "웹훅 URL 확인",
            "설정된 값이 Discord Webhook 주소 형식이 아닙니다. (값은 표시하지 않습니다)",
            "환경 변수/Secrets에 Discord 웹후크 URL이 아닌 값이 들어갔습니다.",
            "Discord 채널 설정에서 복사한 웹후크 URL을 다시 설정하세요.",
        )


def load_results() -> dict:
    """1단계에서 저장한 JSON만 읽는다(API 호출 없음)."""
    if not DATA_FILE.exists():
        raise DiscordError(
            "JSON 읽기",
            f"결과 파일이 없습니다: {DATA_FILE.name}",
            "lck_collector.py를 먼저 실행하지 않았거나 파일 위치가 다릅니다.",
            "python lck_collector.py 실행 후 다시 시도하세요.",
        )
    try:
        data = json.loads(DATA_FILE.read_text(encoding="utf-8"))
    except (ValueError, OSError) as exc:
        raise DiscordError(
            "JSON 읽기",
            redact(str(exc)),
            "JSON 파일이 깨졌거나 읽을 수 없습니다.",
            "lck_collector.py로 결과를 다시 저장하세요.",
        ) from exc

    if not isinstance(data, dict) or not isinstance(data.get("경기결과"), list):
        raise DiscordError(
            "JSON 확인",
            "경기결과 목록을 찾을 수 없습니다.",
            "JSON 구조가 예상과 다릅니다.",
            "lck_match_results.json에 '경기결과' 배열이 있는지 확인하세요.",
        )
    return data

### 7-3. Embed 메시지 만들기

1단계 JSON의 실제 데이터로 대회명·날짜·팀·점수·승리팀·출처 링크를 담습니다. 길이 제한도 검사합니다.

In [ ]:
def cut(text, limit: int) -> str:
    """Discord 글자 제한에 맞게 자른다."""
    text = str(text)
    return text if len(text) <= limit else text[: limit - 1] + "…"


def team_names(match: dict) -> tuple[str, str]:
    teams = match.get("팀", [])
    if isinstance(teams, list) and len(teams) == 2:
        return str(teams[0]), str(teams[1])
    return "팀 정보 없음", "팀 정보 없음"


def make_fields(data: dict) -> list[dict]:
    """경기 1건을 Discord Embed 필드 1개로 바꾼다."""
    fields = []
    for index, match in enumerate(data.get("경기결과", []), 1):
        team1, team2 = team_names(match)
        date = match.get("날짜_KST") or "날짜 확인 불가"
        score = match.get("최종점수") or "점수 정보 없음"
        winner = match.get("승리팀") or "승리팀 정보 없음"
        link = match.get("출처링크") or ""

        name = cut(f"{index}. {date} | {team1} {score} {team2}", MAX_FIELD_NAME_LENGTH)
        value = f"🏆 승리: **{cut(winner, 200)}**"
        if link:
            value += f"\n🔗 [경기 출처]({cut(link, 300)})"
        fields.append({"name": name, "value": cut(value, MAX_FIELD_VALUE_LENGTH), "inline": False})
    return fields


def build_embeds(data: dict) -> list[dict]:
    """필드를 여러 Embed로 나눠 Discord 전체 길이 제한(6000자)을 지킨다."""
    tournament = data.get("대회", {}) if isinstance(data.get("대회"), dict) else {}
    title = cut(f"🏆 {tournament.get('이름', '대회명 정보 없음')} 경기 결과", MAX_TITLE_LENGTH)
    description = cut(
        f"종료일 {tournament.get('종료일', '종료일 정보 없음')} · 최신 {len(data.get('경기결과', []))}건",
        MAX_DESCRIPTION_LENGTH,
    )
    footer_text = cut(data.get("출처_표기", "자료 출처: Liquipedia (CC BY-SA 3.0)"), 200)

    def make_embed(group: list[dict]) -> dict:
        return {
            "title": title,
            "description": description,
            "color": 0x5865F2,  # Discord 블루
            "fields": group,
            "footer": {"text": footer_text},
        }

    def fixed_size() -> int:
        return len(title) + len(description) + len(footer_text)

    embeds = []
    group = []
    group_size = 0
    total_size = 0

    for field in make_fields(data):
        field_size = len(field["name"]) + len(field["value"])
        would_exceed = (
            len(group) >= 5
            or fixed_size() + group_size + field_size > MAX_EMBED_TOTAL
            or total_size + fixed_size() + group_size + field_size > MAX_EMBED_TOTAL
        )
        if group and would_exceed:
            embeds.append(make_embed(group))
            total_size += fixed_size() + group_size
            group, group_size = [], 0
        group.append(field)
        group_size += field_size

    if group:
        embeds.append(make_embed(group))

    if not embeds:
        raise DiscordError(
            "메시지 구성",
            "보낼 경기 결과가 0건입니다.",
            "JSON의 경기결과가 비어 있습니다.",
            "lck_collector.py로 경기 결과를 다시 수집하세요.",
        )
    if len(embeds) > MAX_EMBEDS:
        raise DiscordError(
            "메시지 구성",
            f"Embed가 {len(embeds)}개로 Discord 제한({MAX_EMBEDS}개)을 넘었습니다.",
            "경기 수가 너무 많습니다.",
            "json의 경기결과를 5건 이하로 줄이거나 메시지를 나눠 보내세요.",
        )
    return embeds


def build_payload(data: dict) -> dict:
    """최종 Discord 전송 데이터를 만든다."""
    tournament = data.get("대회", {}) if isinstance(data.get("대회"), dict) else {}
    count = len(data.get("경기결과", []))
    name = tournament.get("이름", "대회명 정보 없음")
    content = cut(f"📢 LCK 경기 결과 알림 · {name} · {count}건", MAX_CONTENT_LENGTH)
    return {"content": content, "embeds": build_embeds(data)}

### 7-4. 전송과 결과 출력

성공/실패 여부를 출력합니다. 401/403/404/429는 즉시 중단하고 우회하지 않습니다.

In [ ]:
def send_webhook(url: str, payload: dict) -> None:
    """웹훅으로 POST 전송한다. 성공/실패 여부를 화면에 출력한다."""
    for attempt in range(DISCORD_MAX_RETRIES + 1):
        try:
            response = requests.post(
                url,
                json=payload,
                headers={"Content-Type": "application/json", "User-Agent": DISCORD_USER_AGENT},
                timeout=DISCORD_TIMEOUT_SECONDS,
            )
        except (requests.Timeout, requests.ConnectionError) as exc:
            if attempt < DISCORD_MAX_RETRIES:
                print(f"  [재시도 {attempt + 1}/{DISCORD_MAX_RETRIES}] 일시적 네트워크 오류")
                time.sleep(RETRY_DELAY_SECONDS * (attempt + 1))
                continue
            raise DiscordError(
                "Discord 전송",
                redact(str(exc), url),
                "일시적인 네트워크 오류가 재시도 후에도 해결되지 않았습니다.",
                "인터넷 연결을 확인하고 잠시 후 다시 실행하세요.",
            ) from exc

        if response.status_code in (200, 204):
            print(f"✅ Discord 전송 성공 (HTTP {response.status_code})")
            print("전송 결과: 성공")
            return

        body = redact(response.text[:300], url)
        status = response.status_code
        if status in (401, 403, 404, 429):
            raise DiscordError(
                "Discord 전송",
                f"HTTP {status}: {body}",
                "인증 실패, 권한 부족, 웹훅 접근 제한 또는 요청 제한입니다.",
                "웹훅 URL과 채널 권한을 확인하고 잠시 후 다시 시도하세요. 우회 시도는 하지 마세요.",
            )
        if 500 <= status <= 599 and attempt < DISCORD_MAX_RETRIES:
            print(f"  [재시도 {attempt + 1}/{DISCORD_MAX_RETRIES}] Discord 서버 오류 (HTTP {status})")
            time.sleep(RETRY_DELAY_SECONDS * (attempt + 1))
            continue
        if status == 400:
            raise DiscordError(
                "Discord 전송",
                f"HTTP {status}: {body}",
                "메시지 형식이 Discord 제한을 넘었습니다.",
                "Embed 길이와 개수를 줄이세요(본문 2000자, Embed 합계 6000자).",
            )
        raise DiscordError(
            "Discord 전송",
            f"HTTP {status}: {body}",
            "Discord가 요청을 처리하지 못했습니다.",
            "상태 코드를 확인하고 잠시 후 다시 실행하세요.",
        )


def discord_main():
    """URL 확인 -> JSON 읽기 -> 메시지 구성 -> 전송."""
    url = get_webhook_url()
    if not url:
        print("❌ Discord 전송을 건너뜁니다. 웹훅 URL이 설정되지 않았습니다.")
        print_setup_guide()
        return None  # 실패도 성공도 아닌 '미전송'

    try:
        validate_webhook_url(url)
        data = load_results()
        payload = build_payload(data)
        print(
            f"메시지 구성 완료: 본문 {len(payload['content'])}자 / "
            f"Embed {len(payload['embeds'])}개 / 경기 {len(data.get('경기결과', []))}건"
        )
        send_webhook(url, payload)
        return True
    except DiscordError as error:
        discord_print_error_report(error, url)
        return False


def discord_notebook_main():
    """Colab 노트북에서 호출하는 진입점."""
    return discord_main()

In [ ]:
discord_result = discord_notebook_main()

# True=전송 성공, None=URL 미설정으로 건너뜀, False=전송 실패
print('Discord 최종 상태:', {True: '성공', None: '건너뜀', False: '실패'}[discord_result])